# Imports

In [1]:
import torch
import sys
import os
from torch import Tensor
sys.path.append('..')
import pandas as pd
import plotly.express as px
import plotly.graph_objs as go
from definitions import DATA_DIR, EXPERIMENTS_DIR, ROOT_DIR
from src.commands.train import Trainer, Config, ArticleAsKG
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    hamming_loss,
)
from PIL import ImageColor

In [2]:
px.defaults.template = 'seaborn'

In [3]:
dir = os.path.join(ROOT_DIR, 'assets', 'articles_classification_results')

# Load the results

In [4]:
# TASK = 'hierarchical_classification'
EXPERIMENT = 'nantes-uni-2020'
CORPUS = 'nantes-uni-2020-2'
EXPERIMENT_DIR = os.path.join(EXPERIMENTS_DIR, EXPERIMENT)
RESULTS_DIR = os.path.join(EXPERIMENT_DIR, 'results')
HISTORIES_DIR = os.path.join(EXPERIMENT_DIR, 'histories')
CONFIG_PATH = os.path.join(EXPERIMENT_DIR, 'config.json')
SPLIT_PATH = os.path.join(EXPERIMENT_DIR, 'split_sub_dsis_4.pt')

In [5]:
history = pd.concat([
    pd.read_csv(os.path.join(HISTORIES_DIR, filename))
    for filename in os.listdir(HISTORIES_DIR)
])
history = history \
    .drop(columns = ['iteration']) \
    .groupby(['epoch','phase','exp','trial']) \
    .mean() \
    .reset_index()
history.rename(columns = { "exp" : "expr" }, inplace = True)
history.head()

,epoch,phase,expr,trial,Unnamed: 0,guidance_loss,loss,guidance_acc,guidance_f1
0,0,train,Attn_Attn_null_False,0,120.5,0.187430,0.187430,0.927456,0.278295
1,0,train,Attn_Attn_null_False,1,120.5,0.188121,0.188121,0.928504,0.278145
2,0,train,Attn_Attn_null_False,2,120.5,0.184909,0.184909,0.929709,0.291059
3,0,train,Attn_Attn_null_False,3,120.5,0.186446,0.186446,0.928762,0.278922
4,0,train,Attn_Attn_null_True,0,120.5,0.341983,0.341983,0.850733,0.852195


In [6]:
history_mean = history \
    .drop(columns = 'trial') \
    .groupby(['epoch','phase','expr']).mean() \
    .add_suffix('_mean')

history_std = history \
    .drop(columns = 'trial') \
    .groupby(['epoch','phase','expr']).std() \
    .add_suffix('_std')

In [7]:
history = history_mean.join(other = history_std).reset_index()
history.head()

,epoch,phase,expr,Unnamed: 0_mean,guidance_loss_mean,loss_mean,guidance_acc_mean,guidance_f1_mean,Unnamed: 0_std,guidance_loss_std,loss_std,guidance_acc_std,guidance_f1_std
0,0,train,Attn_Attn_null_False,120.5,0.186726,0.186726,0.928608,0.281605,0.0,0.001393,0.001393,0.000926,0.006311
1,0,train,Attn_Attn_null_True,120.5,0.343446,0.343446,0.847250,0.848257,0.0,0.003310,0.003310,0.003107,0.002946
2,0,train,Attn_Max_null_False,120.5,0.197663,0.197663,0.923399,0.253837,0.0,0.001213,0.001213,0.000594,0.006884
3,0,train,Attn_Max_null_True,120.5,0.358747,0.358747,0.839698,0.840458,0.0,0.006085,0.006085,0.004563,0.003035
4,0,train,Attn_Mean_null_False,120.5,0.187940,0.187940,0.927675,0.278082,0.0,0.001078,0.001078,0.000221,0.001991


# Load Data

In [8]:
config = Config(experiment=EXPERIMENT)
trainer = Trainer(experiment=config.experiment, config=config.training_config)
configs = trainer.get_configs()

In [9]:
data = trainer.load_data()
data = trainer.filter_data(data)
data = trainer.keep_largest_cc(data) if trainer.config.keep_largest_cc else data
data.split = trainer.create_or_load_split(data)
data.articles.split = data.split

In [13]:
metadata = data.metadata

# Learning Curves

In [14]:
def line_plot(
    df : pd.DataFrame,
    metric : str
) -> None:
    
    fig = go.Figure()

    colors = px.colors.qualitative.Plotly
    exprs = df['expr'].unique()

    for i, expr in enumerate(exprs):

        history_val = df.query(f"phase == 'val' and expr == '{expr}'")
        color = colors[i % len(colors)]

        fig.add_trace(
            go.Scatter(
                name = expr,
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean'],
                mode='lines',
                line=dict(color=color),
            )
        )

        fig.add_trace(
            go.Scatter(
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean']+history_val[f'{metric}_std'],
                mode='lines',
                marker=dict(color="#444"),
                line=dict(width=0),
                showlegend=False
            )
        )

        fig.add_trace(
            go.Scatter(
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean']-history_val[f'{metric}_std'],
                marker=dict(color="#444"),
                line=dict(width=0),
                mode='lines',
                fillcolor=f'rgb{ImageColor.getcolor(color, "RGB")}'.replace('rgb','rgba').replace(')',',0.2)'),
                fill='tonexty',
                showlegend=False
            )
        )

    fig.update_layout(
        xaxis_title='Epoch',
        yaxis_title=metric.capitalize(),
        width = 960,
        height = 800
    )

    fig.show()

In [15]:
line_plot(history, 'loss')

In [16]:
line_plot(history, 'guidance_acc')

In [17]:
line_plot(history, 'guidance_f1')

# Calculate other metrics

## Load Predictions

In [18]:
def load_results(results_dir : str) -> dict[tuple[str, str], Tensor]:

    results = dict()

    for filename in os.listdir(results_dir):

        parts = filename.split('.')[0].split('_')
        trial = parts[-1]
        expr = '_'.join(parts[1:-1])

        results[(expr,trial)] = torch.load(os.path.join(RESULTS_DIR, filename), map_location='cpu', weights_only=False)

    return results

In [19]:
results = load_results(RESULTS_DIR)

## Label Wise & Row Wise Metrics

In [20]:
def display_mean_std_df(df : pd.DataFrame) -> pd.DataFrame:

    columns = set([
        col.replace('_mean','')
        for col in df.columns if 'mean' in col
    ])

    new_df = pd.DataFrame()

    for col in df.columns:

        if not col.endswith('_mean') and not col.endswith('_std'):
            new_df[col] = df[col]

    for col in columns:
        new_df[col] = df[f'{col}_mean'].apply(lambda x : f'{x:.2%}') + ' ± ' + df[f'{col}_std'].apply(lambda x : f'{x:.2%}')

    return new_df

In [21]:
def get_experiment_params(expr : str) -> dict:

    parts = expr.split('_')
        
    return {
        "GNN Variant" : parts[0],
        "Readout Function" : parts[1],
        "Pooling Layer" : parts[2],
        "Undersampling" : parts[3]
    }

In [22]:
def get_label_wise_metrics(
    data : ArticleAsKG,
    results : dict[tuple[str,str],Tensor],
    labels : list[str]
) -> tuple[pd.DataFrame,pd.DataFrame]:

    label_wise_metrics = []

    for label in range(data.y.size(-1)):
            
        for (expr,trial),result in results.items():

            y = data.y[data.split == int(trial)]
            y_hat = result[data.split == int(trial)].sigmoid()
            y_hard = y_hat.gt(0.5).long()
            
            label_wise_metrics.append({
                "team" : label,
                "expr" : expr,
                "trial" : trial,
                "accuracy" : accuracy_score(y[:,label], y_hard[:,label]),
                "f1_score" : f1_score(y[:,label], y_hard[:,label]),
                "precision" : precision_score(y[:,label], y_hard[:,label]),
                "recall" : recall_score(y[:,label], y_hard[:,label]),
                "auc" : roc_auc_score(y[:,label], y_hat[:,label])
            })

    label_wise_metrics = pd.DataFrame(label_wise_metrics)
    label_wise_metrics['team'] = label_wise_metrics['team'].apply(labels.__getitem__)
    label_wise_metrics['expr'] = label_wise_metrics['expr'].apply(str)

    label_wise_metrics_agg = label_wise_metrics.drop(columns="team").groupby(["expr","trial"]).mean().reset_index().sort_values(by = 'expr')

    label_wise_metrics_mean = label_wise_metrics_agg \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .mean().add_suffix('_mean')

    label_wise_metrics_std = label_wise_metrics_agg \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .std().add_suffix('_std')

    label_wise_metrics_agg = label_wise_metrics_mean.join(label_wise_metrics_std).reset_index()

    return label_wise_metrics_agg,label_wise_metrics

In [23]:
def get_row_wise_metrics(
    data : ArticleAsKG,
    results : dict[tuple[str,str],Tensor],
) -> tuple[pd.DataFrame,pd.DataFrame]:

    row_wise_metrics = []

    for (expr,trial),result in results.items():

        y = data.y[data.split == int(trial)]
        y_hat = result[data.split == int(trial)].sigmoid()
        y_hard = y_hat.gt(0.5).long()

        row_wise_metrics.append({
            "expr" : expr,
            "trial" : trial,
            "accuracy" : accuracy_score(y, y_hard),
            "f1_score" : f1_score(y, y_hard , average='macro'),
            "precision" : precision_score(y, y_hard , average='macro'),
            "recall" : recall_score(y, y_hard , average='macro'),
            "auc" : roc_auc_score(y, y_hat , average='macro'),
            "hamming_loss" : hamming_loss(y, y_hard)
        })

    row_wise_metrics = pd.DataFrame(row_wise_metrics)

    row_wise_metrics_mean = row_wise_metrics \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .mean().add_suffix('_mean')

    row_wise_metrics_std = row_wise_metrics \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .std().add_suffix('_std')

    row_wise_metrics_agg = row_wise_metrics_mean.join(row_wise_metrics_std).reset_index()

    return row_wise_metrics_agg,row_wise_metrics

In [24]:
label_wise_metrics_agg,label_wise_metrics = get_label_wise_metrics(data.articles, results, metadata['labels'])
display_mean_std_df(label_wise_metrics_agg)

,expr,recall,accuracy,f1_score,precision,auc
0,Attn_Attn_null_False,65.77% ± 1.59%,95.50% ± 0.10%,71.10% ± 0.65%,78.61% ± 1.71%,96.43% ± 0.23%
1,Attn_Attn_null_True,77.08% ± 0.61%,94.46% ± 0.19%,70.77% ± 0.89%,66.14% ± 1.84%,96.70% ± 0.17%
2,Attn_Max_null_False,62.97% ± 1.39%,95.42% ± 0.20%,69.86% ± 1.04%,81.88% ± 3.13%,96.37% ± 0.36%
3,Attn_Max_null_True,75.24% ± 2.03%,94.43% ± 0.10%,70.94% ± 0.38%,68.02% ± 2.31%,96.52% ± 0.10%
4,Attn_Mean_null_False,59.57% ± 1.67%,95.40% ± 0.21%,67.62% ± 1.55%,82.67% ± 1.60%,96.59% ± 0.18%
5,Attn_Mean_null_True,75.24% ± 2.70%,94.41% ± 0.25%,70.28% ± 0.73%,67.13% ± 3.03%,96.82% ± 0.12%
6,Conv_Attn_null_False,66.88% ± 2.91%,95.49% ± 0.15%,71.53% ± 1.52%,78.15% ± 2.59%,96.45% ± 0.21%
7,Conv_Attn_null_True,77.63% ± 1.69%,94.63% ± 0.15%,71.38% ± 0.62%,66.92% ± 1.19%,96.72% ± 0.20%
8,Conv_Max_null_False,63.82% ± 3.57%,95.40% ± 0.19%,69.81% ± 2.21%,79.58% ± 2.29%,96.37% ± 0.40%
9,Conv_Max_null_True,72.80% ± 3.41%,94.58% ± 0.16%,70.21% ± 0.59%,69.14% ± 3.29%,96.71% ± 0.09%


In [25]:
row_wise_metrics_agg,row_wise_metrics = get_row_wise_metrics(data.articles, results)
metrics = display_mean_std_df(row_wise_metrics_agg)
metrics

,expr,recall,accuracy,f1_score,precision,hamming_loss,auc
0,Attn_Attn_null_False,65.77% ± 1.59%,50.85% ± 1.47%,71.10% ± 0.65%,78.61% ± 1.71%,4.50% ± 0.10%,96.43% ± 0.23%
1,Attn_Attn_null_True,77.08% ± 0.61%,42.22% ± 1.00%,70.77% ± 0.89%,66.14% ± 1.84%,5.54% ± 0.19%,96.70% ± 0.17%
2,Attn_Max_null_False,62.97% ± 1.39%,49.50% ± 1.59%,69.86% ± 1.04%,81.88% ± 3.13%,4.58% ± 0.20%,96.37% ± 0.36%
3,Attn_Max_null_True,75.24% ± 2.03%,41.49% ± 0.72%,70.94% ± 0.38%,68.02% ± 2.31%,5.57% ± 0.10%,96.52% ± 0.10%
4,Attn_Mean_null_False,59.57% ± 1.67%,49.96% ± 1.11%,67.62% ± 1.55%,82.67% ± 1.60%,4.60% ± 0.21%,96.59% ± 0.18%
5,Attn_Mean_null_True,75.24% ± 2.70%,41.24% ± 1.70%,70.28% ± 0.73%,67.13% ± 3.03%,5.59% ± 0.25%,96.82% ± 0.12%
6,Conv_Attn_null_False,66.88% ± 2.91%,50.85% ± 1.45%,71.53% ± 1.52%,78.15% ± 2.59%,4.51% ± 0.15%,96.45% ± 0.21%
7,Conv_Attn_null_True,77.63% ± 1.69%,43.77% ± 0.59%,71.38% ± 0.62%,66.92% ± 1.19%,5.37% ± 0.15%,96.72% ± 0.20%
8,Conv_Max_null_False,63.82% ± 3.57%,49.58% ± 0.79%,69.81% ± 2.21%,79.58% ± 2.29%,4.60% ± 0.19%,96.37% ± 0.40%
9,Conv_Max_null_True,72.80% ± 3.41%,41.90% ± 0.91%,70.21% ± 0.59%,69.14% ± 3.29%,5.42% ± 0.16%,96.71% ± 0.09%


In [26]:
hparams = metrics.apply(lambda x : pd.Series(get_experiment_params(x['expr'])), axis = 1)
metrics = hparams.join(metrics, lsuffix="hparams")
metrics['Experiment'] = metrics['expr']
metrics = metrics.drop(columns = ['expr'])
metrics = metrics[[metrics.columns[-1], *metrics.columns[:-1]]]
# metrics.to_csv(os.path.join(dir, "hierarchical_teams_metrics.xlsx"), index = False)
metrics

,Experiment,GNN Variant,Readout Function,Pooling Layer,Undersampling,recall,accuracy,f1_score,precision,hamming_loss,auc
0,Attn_Attn_null_False,Attn,Attn,null,False,65.77% ± 1.59%,50.85% ± 1.47%,71.10% ± 0.65%,78.61% ± 1.71%,4.50% ± 0.10%,96.43% ± 0.23%
1,Attn_Attn_null_True,Attn,Attn,null,True,77.08% ± 0.61%,42.22% ± 1.00%,70.77% ± 0.89%,66.14% ± 1.84%,5.54% ± 0.19%,96.70% ± 0.17%
2,Attn_Max_null_False,Attn,Max,null,False,62.97% ± 1.39%,49.50% ± 1.59%,69.86% ± 1.04%,81.88% ± 3.13%,4.58% ± 0.20%,96.37% ± 0.36%
3,Attn_Max_null_True,Attn,Max,null,True,75.24% ± 2.03%,41.49% ± 0.72%,70.94% ± 0.38%,68.02% ± 2.31%,5.57% ± 0.10%,96.52% ± 0.10%
4,Attn_Mean_null_False,Attn,Mean,null,False,59.57% ± 1.67%,49.96% ± 1.11%,67.62% ± 1.55%,82.67% ± 1.60%,4.60% ± 0.21%,96.59% ± 0.18%
5,Attn_Mean_null_True,Attn,Mean,null,True,75.24% ± 2.70%,41.24% ± 1.70%,70.28% ± 0.73%,67.13% ± 3.03%,5.59% ± 0.25%,96.82% ± 0.12%
6,Conv_Attn_null_False,Conv,Attn,null,False,66.88% ± 2.91%,50.85% ± 1.45%,71.53% ± 1.52%,78.15% ± 2.59%,4.51% ± 0.15%,96.45% ± 0.21%
7,Conv_Attn_null_True,Conv,Attn,null,True,77.63% ± 1.69%,43.77% ± 0.59%,71.38% ± 0.62%,66.92% ± 1.19%,5.37% ± 0.15%,96.72% ± 0.20%
8,Conv_Max_null_False,Conv,Max,null,False,63.82% ± 3.57%,49.58% ± 0.79%,69.81% ± 2.21%,79.58% ± 2.29%,4.60% ± 0.19%,96.37% ± 0.40%
9,Conv_Max_null_True,Conv,Max,null,True,72.80% ± 3.41%,41.90% ± 0.91%,70.21% ± 0.59%,69.14% ± 3.29%,5.42% ± 0.16%,96.71% ± 0.09%


- Visualizations

In [27]:
def bar_plot(df : pd.DataFrame, metric : str) -> None:

    fig = px.bar(
        data_frame = df.sort_values(by = f'{metric}_mean'),
        x = 'expr',
        y = f'{metric}_mean',
        barmode = 'group',
        error_y = f'{metric}_std',
        text = f'{metric}_mean',
        width = 950,
        height = 800
    )

    fig.update_layout(
        xaxis_title = 'Experiment',
        yaxis_title = metric.capitalize().replace('_',' '),
    )

    fig.update_traces(
        textposition = 'outside',
        texttemplate = '%{text:.2%}'
    )

    fig.show()

In [28]:
bar_plot(label_wise_metrics_agg, 'accuracy')

In [29]:
bar_plot(label_wise_metrics_agg, 'f1_score')

In [30]:
bar_plot(row_wise_metrics_agg, 'accuracy')